In [2]:
#1. section ini berisi import dan data mentah
import numpy as np
import pandas as pd

raw_data = pd.read_csv("D:/Documents/Projectku/Data Analyst/Python/Projek Club Member/python-club_member_info.csv")
df = pd.DataFrame(raw_data)

#membuat backup data
club = df.copy()

In [15]:
#2.  section ini berisi Data Understanding
print("1. Ambil 5 baris data sebagai ringkasan\n\n", club.head(), "\n\n")
print("2. Struktur data secara umum", club.info(), "\n\n")
print("3. Struktur data per kolom\n\n", club.count(), "\n\n")
print("4. Struktur data kolom Age (Numerikal)\n\n", club.describe(), "\n\n")
print("5. Struktur data untuk kolom martial status\n\n", club["martial_status"].value_counts(dropna=False))

1. Ambil 5 baris data sebagai ringkasan

                full_name   age martial_status                    email  \
0             addie lush  40.0        married    alush0@shutterfly.com   
1           ROCK CRADICK  46.0        married   rcradick1@newsvine.com   
2      ???Sydel Sharvell  46.0       divorced  ssharvell2@amazon.co.jp   
3  Constantin de la cruz  35.0            NaN        co3@bloglines.com   
4         Gaylor Redhole  38.0        married   gredhole4@japanpost.jp   

          phone                                  full_address  \
0  254-389-8708               3226 Eastlawn Pass,Temple,Texas   
1  910-566-2007  4 Harbort Avenue,Fayetteville,North Carolina   
2  702-187-8715               4 School Place,Las Vegas,Nevada   
3  402-688-7162            6 Monument Crossing,Omaha,Nebraska   
4  917-394-6001       88 Cherokee Pass,New York City,New York   

                     job_title membership_date  
0          Assistant Professor       7/31/2013  
1               Programm

Berdasarkan bagian #2, secara garis besar dataset ini memiliki beberapa eror mulai dari NA, kolom membership_date yang masih string, hingga penulisan pekerjaan yang salah. Oleh karena itu, ada section #4 untuk memperjelas error pada tiap kolom.
Selain itu, masih belum ada kolom ID sebagai key primer, maka harus ditambah terlebih dahulu.
Berikut adalah ketentuan seberapa penting data pada kolom.
    1. Mandatory field -- bagian yang wajib diisi, data NA akan menyebabkan baris dihapus
        - email -- sebagai unique identifier
        - membership_date
    2. optional field -- bagian yang tidak mewajibkan diisi, data NA masih bisa dibiarkan. Harus menghubungi klien untuk data kosongnya.
        - full_name
        - age
        - martial_status
        - phone
        - full_address
        - job title

In [4]:
#3. penambahan kolom ID
club["id"] = range(1, len(club)+1)
club = club.set_index("id")
print(club.tail())

             full_name   age martial_status                      email  \
id                                                                       
2006     Gasper Halden  62.0      separated    ghaldenrn@bigcartel.com   
2007  netty hammelberg  67.0         single  nhammelbergro@weather.com   
2008   Phillip Piddick  40.0        married        ppiddickrp@hibu.com   
2009      ALLIX LAWRIE  50.0       divorced          alawrierq@wsj.com   
2010   rockey gimbrett  26.0        married      rgimbrettrr@google.ca   

             phone                           full_address  \
id                                                          
2006  571-930-5137      406 Shelley Lane,Ashburn,Virginia   
2007  423-943-8510  376 Dapin Place,Kingsport,Tennesseeee   
2008  512-886-9162  767 Burning Wood Parkway,Austin,Texas   
2009  515-452-7385           162 Orin Way,Des Moines,Iowa   
2010  713-436-2805       77 Dorton Crossing,Houston,Texas   

                     job_title membership_date  
id  

#4. Section data error identification
secara umum saya akan mengecek tiap kolom untuk error:
    - data NA
    - karakter aneh -- kecuali untuk email
    - string kosong ('')
berikut untuk kolom khusus
    - range validation -- khusus data age dan membership_date (sesuai kebutuhan)
    - invalid format -- sudah dicek di section #2, hanya kolom membership_date
    - Categorical Value Validation -- khusus kolom martial_status
    - data duplikat -- khusus kolom email
    - cek error membership date tanggalnya lebih dari sekarang dan dibawah 2000

In [18]:
#4.1 full_name
fn_cek_na = club["full_name"].isna().sum()
fn_cek_strange = club["full_name"].str.contains(r"[^a-zA-Z' -]")
print(f"total baris NA: {fn_cek_na}")
print(f"total baris dengan karakter aneh (bukan huruf/strip/apostrof): {fn_cek_strange.sum()}")
print(club[fn_cek_strange])

total baris NA: 0
total baris dengan karakter aneh (bukan huruf/strip/apostrof): 52
                 full_name   age martial_status  \
id                                                
3        ???Sydel Sharvell  46.0       divorced   
25          ???Zara Brandi  33.0       divorced   
35            ???Bent Gipp  53.0        married   
59         ???Harold Fritz  50.0        married   
85          ???Dilan Straw  32.0        married   
86          ???Glen Levene  48.0        married   
124          ???Mord Nagle  32.0         single   
130           ???Kris Pole  47.0         single   
187          ???Mil Croose  33.0         single   
234        ???Reggi Pantry  27.0      separated   
258        ???Patti Corker  50.0        married   
278         ???Katrina Igo  35.0        married   
325          ???Lou Ovitts  52.0        married   
341      ???Eberto Cordrey  31.0         single   
367         ???Ahmed Elmar  54.0        married   
401            ???Zed Tear  49.0        married  

In [18]:
#4.2 age
age_cek_na = club["age"].isna().sum()
#age_cek_strange = club["age"].astype("str").str.match(r"^\d+$", na=False).count()
age_cek_valid = club["age"] > 100
print(f"total baris NA: {age_cek_na}")
#print(f"total baris dengan karakter aneh: {age_cek_strange}")
print(f"total baris dengan usia di atas 100 tahun: {age_cek_valid.sum()}")

total baris NA: 3
total baris dengan usia di atas 100 tahun: 15


In [19]:
#4.3 martial status
ms_cek_na = club["martial_status"].isna().sum()
ms_cek_catvalid = club["martial_status"].value_counts(dropna=False)
print(f"jumlah data NA: {ms_cek_na}")
print(f"\ndistribusi kategorikal:\n{ms_cek_catvalid}")
print("\nada kesalahan pengisian di mana divored harusnya divorced")

jumlah data NA: 20

distribusi kategorikal:
martial_status
married      881
single       656
divorced     282
separated    167
NaN           20
divored        4
Name: count, dtype: int64

ada kesalahan pengisian di mana divored harusnya divorced


In [20]:
#4.4 email: melihat data duplikat
email_cek_na = club["email"].isna().sum()
email_cek_duplikat = club.duplicated(subset="email", keep=False).sum()
print(f"total baris NA: {email_cek_na} baris")
print(f"total baris duplikat: {email_cek_duplikat} baris")

total baris NA: 0 baris
total baris duplikat: 19 baris


In [21]:
#4.5 phone
phone_cek_na = club["phone"].isna().sum()
phone_cek_strange = club["phone"].str.contains(r"[^0-9+\-\s()]").sum()
phone_kar_lebih = (club["phone"].str.len() < 8) | (club["phone"].str.len() > 14)
print(f"total baris NA: {phone_cek_na} baris")
print(f"total baris karakter nomor hp aneh: {phone_cek_strange} baris")
print(f"total baris phone yang lebih invalid (8 digit < karakter > 14 digit): {phone_kar_lebih.sum()} baris")

total baris NA: 9 baris
total baris karakter nomor hp aneh: 0 baris
total baris phone yang lebih invalid (8 digit < karakter > 14 digit): 0 baris


In [22]:
#4.6 full_address
fa_cek_na = club["full_address"].isna().sum()
fa_cek_strange = club["full_address"].str.contains(r"[^a-zA-Z0-9\s.,+/#-]").sum()
print(f"total baris NA: {fa_cek_na} baris")
print(f"total baris dengan karakter aneh: {fa_cek_strange} baris")

total baris NA: 0 baris
total baris dengan karakter aneh: 0 baris


In [23]:
#4.7 job_title
jt_cek_na = club["job_title"].isna().sum()
jt_cek_strange = club["full_address"].str.contains(r"[I|II|III|IV]").sum()
print(f"total baris NA: {jt_cek_na} baris")
print(f"total baris dengan karakter aneh: {jt_cek_strange} baris")

total baris NA: 39 baris
total baris dengan karakter aneh: 380 baris


In [47]:
#4.8 membership_date -- terlebih dahulu ubah ke format tanggal
md_cek_nontanggal = club["membership_date"].astype("str").str.contains(r"[^0-9\-/]").sum()
club["membership_date"] = pd.to_datetime(club["membership_date"])

md_cek_na = club["membership_date"].isna().sum()
md_cek_invalid = club["membership_date"] > pd.Timestamp.today()
md_cek_lama = club["membership_date"].dt.year < 2000

print(f"Jumlah baris dengan data bukan tanggal: {md_cek_nontanggal}")
print(f"jumlah baris dengan tanggal aneh (lebih dari hari sekarang): {md_cek_invalid.sum()}")
print(f"jumlah baris dengan tanggal aneh (di bawah tahun 2000): {md_cek_lama.sum()}")
print(f"total baris NA: {md_cek_na} baris")
#print(club[md_cek_lama])
print(club["membership_date"].dt.year.value_counts().sort_index())

Jumlah baris dengan data bukan tanggal: 0
jumlah baris dengan tanggal aneh (lebih dari hari sekarang): 0
jumlah baris dengan tanggal aneh (di bawah tahun 2000): 16
total baris NA: 0 baris
membership_date
1912      3
1913      1
1914      1
1915      5
1916      2
1917      1
1919      1
1921      2
2012     89
2013     84
2014    103
2015    238
2016    232
2017    246
2018    233
2019    237
2020    205
2021    226
2022     91
Name: count, dtype: int64


Penjelasan dari tahap identification Error
    1. full_name: karakter aneh non huruf/strip/apostrof termasuk spasi 52. Nama masih belum terstandarisasi (lowercase, capitalize dll). selain itu ada spasi di awal kalimat
    2. age: ada 3 baris NA, ada data yang diluar range validation berusia 100 tahun sebanyak 15, disebabkan oleh penulisan angka yang ganda seperti 455 harusnya 45.
    3. martial_status: ada 20 data NA dan kesalahan penulisan kategori harusnya divorced malah divorced. Selain itu ada kesalahan penulisan nama kolom harusnya marital_status
    4. email: ada 19 baris data yang ternyata duplikat
    5. phone: ada 9 baris NA
    6. full_address: hanya pemisahan menjadi 3 kolom yaitu (street_address, city, state)
    7. job_title: ada 39 baris NA, dan 380 baris dengan karakter aneh (huruf romawi I,II,III,IV)
    8. membership_date: ada tanggal invalid, di mana total 16 data ada di tahun 1900an

perbaikan yang akan dilakukan:
    1. full_name: menghapus karakter aneh, standarisasi ke kapital tiap huruf awal kata, dan hapus spasi berlebih
    2. age: karena ada penulisan angka ganda dengan pola berulang, maka akan direplace menggunakan regex
    3. martial_status: mengganti kata divored menjadi divorced
    4. email: menghapus data duplikat
    5. phone: tidak ada perbaikan (sesuai dengan aturan pada section #2)
    6. full_address: memisahkan kolom
    7. job_title: menghapus karakter huruf romawi pada string
    8. membership_date: perbaikan dengan menambah tahun +100, sehingga menjadi tahun 2010+

#5. Section data cleaning, terbagi menjadi 4 tahap:
    tahap 1. duplicate handling (email)
    tahap 2. missing value handling ()
    tahap 3. standarisasi data (full_name, full_address)
    tahap 4. invalid value check (martial_status, age)

In [57]:
clubku = club.copy()

In [32]:
#5.1 duplicate handling
club = club.drop_duplicates(subset="email", keep="first")
data_duplikat = club.duplicated(subset="email", keep=False)
print(f"jumlah data duplikat sekarang: {data_duplikat.sum()}")

jumlah data duplikat sekarang: 0


In [ ]:
#full_name
fn_strange = club["full_name"].str.contains(r"[^a-zA-Z' -]")
club.loc[fn_strange, "full_name"] = club.loc[fn_strange, "full_name"].str.replace(r"[^a-zA-Z' -]","", regex=True)
club["full_name"] = club["full_name"].str.title()
club["full_name"] = club["full_name"].str.strip()

print(f"jumlah karakter aneh sekarang: {fn_strange.sum()}")
print(club["full_name"].sample(10))

jumlah karakter aneh sekarang: 0
id
1506          Caye Crips
570        Boyd Armfirld
1352       Mannie Peiser
1146       Sheena Sporle
507      Felicle Loffill
935     Simona Josefsson
148     Kanya Gilphillan
2000      Roxine Zorzini
767         Herve Divine
1261       Sayers Rapson
Name: full_name, dtype: object


In [ ]:
#age
club["age"] = club["age"].astype("Int64") #ubah ke format integral bukan float
age_ganda = club["age"].astype(str).str.contains(r"\d{3}")
club.loc[age_ganda, "age"] = club.loc[age_ganda,"age"].astype(str).str[:2].astype(float)
print(age_ganda.sum())

0


In [37]:
#marital_status
club.rename(columns={"martial_status":"marital_status"}, inplace=True)
ms_divored = club["marital_status"].str.contains("divored", na=False)
club.loc[ms_divored, "marital_status"] = club.loc[ms_divored, "marital_status"].str.replace("divored", "divorced")
print(ms_divored.sum())

0


In [38]:
#full_address
club[["street_address", "city", "state"]] = club["full_address"].str.split(",", expand=True)
# Reorder kolom
club = club[["full_name", "age", "marital_status", "email", "phone", "full_address", "street_address", "city", "state", "job_title", "membership_date"]]
print(club.head())

                full_name   age marital_status                    email  \
id                                                                        
1              Addie Lush  40.0        married    alush0@shutterfly.com   
2            Rock Cradick  46.0        married   rcradick1@newsvine.com   
3          Sydel Sharvell  46.0       divorced  ssharvell2@amazon.co.jp   
4   Constantin De La Cruz  35.0            NaN        co3@bloglines.com   
5          Gaylor Redhole  38.0        married   gredhole4@japanpost.jp   

           phone                                  full_address  \
id                                                               
1   254-389-8708               3226 Eastlawn Pass,Temple,Texas   
2   910-566-2007  4 Harbort Avenue,Fayetteville,North Carolina   
3   702-187-8715               4 School Place,Las Vegas,Nevada   
4   402-688-7162            6 Monument Crossing,Omaha,Nebraska   
5   917-394-6001       88 Cherokee Pass,New York City,New York   

         st

In [42]:
#job_title
jt_strange = club["job_title"].str.contains(r"[I|II|III|IV]$", na=False)
club.loc[jt_strange, "job_title"] = club.loc[jt_strange, "job_title"].str.replace(r"\s*(I|II|III|IV)$","", regex=True)
print(f"jumlah karakter aneh sekarang: {jt_strange.sum()}")

jumlah karakter aneh sekarang: 0


In [66]:
#membership_date
md_lama = club["membership_date"].dt.year < 2000
club.loc[md_lama, "membership_date"] = club.loc[md_lama, "membership_date"] + pd.DateOffset(years=100)
print(club[md_lama])

Empty DataFrame
Columns: [full_name, age, marital_status, email, phone, full_address, street_address, city, state, job_title, membership_date]
Index: []


Perbaikan sudah selesai!
sekarang saya akan ekspor dan memberi nama file clean-club_member_info

In [76]:
club.to_csv("clean-club_member_info.csv")